In [1]:
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
from IPython.display import clear_output
from torch.utils.data import DataLoader
from tqdm import tqdm

from kolmogorov_flow_matching.dataset import NormalizedKolmogorovDataset
from kolmogorov_flow_matching.models.diffusion import ConditionalDiffusion
from kolmogorov_flow_matching.models.unet import ConditionalUnetBackbone
from kolmogorov_flow_matching.source import download_train_set

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
print("Downloading/Locating dataset...")
train_h5_path = download_train_set("../data/train.h5")

Downloading/Locating dataset...


In [3]:
# 1. Configuration
k_frames = 4  # Number of history frames to condition on

# The dataset will automatically compute and print the mean/std on initialization
train_dataset = NormalizedKolmogorovDataset(
    h5_file_path=train_h5_path, k_frames=k_frames, dataset_key="train/u"
)

# data_shape is [C, H, W] -> For Kolmogorov flow, C=1 usually. H/W depend on the dataset (e.g., 64x64).
# Update data_shape dynamically based on what the dataset returns
sample_batch = train_dataset[0]
c, h, w = sample_batch["target"].shape

print(f"Detected target shape: Channels={c}, Height={h}, Width={w}")

backbone = ConditionalUnetBackbone(data_shape=[c, h, w], k_frames=k_frames).to(device)

# The diffusion model inherits the normalization stats from the dataset
model = ConditionalDiffusion(
    backbone=backbone, mean=train_dataset.mean, std=train_dataset.std, T=1000
).to(device)

Computing global statistics for ../data/train.h5/train/KolmFlow_train_1024.h5...
Done! Mean: 0.00000, Std: 3.50582
Detected target shape: Channels=1, Height=160, Width=160


In [4]:
num_epochs = 5
batch_size = 16
learning_rate = 1e-4
max_grad_norm = 1.0

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True
)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

print("Starting training...")
epoch_history = []
loss_history = []
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    # The 'with' statement ensures the bar safely closes before the print statement
    with tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}") as progress_bar:
        for batch in progress_bar:
            # Move data to GPU
            x_cond = batch["condition"].to(device)
            x_target = batch["target"].to(device)

            # Zero the gradients
            optimizer.zero_grad()

            # Forward pass
            loss = model.get_training_loss(x_target, x_cond)

            # Backward pass
            loss.backward()

            # Update weights
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
            optimizer.step()

            # Update progress bar statistics
            epoch_loss += loss.item()
            progress_bar.set_postfix({"Loss": f"{loss.item():.5f}"})

    avg_loss = epoch_loss / len(train_loader)

    # 1. Store the metrics
    epoch_history.append(epoch + 1)
    loss_history.append(avg_loss)

    # 2. Clear the cell output (wait=True prevents flickering)
    clear_output(wait=True)

    # 3. Draw the updated plot
    plt.figure(figsize=(10, 5))
    plt.plot(epoch_history, loss_history, marker="o", linestyle="-", color="b")
    plt.title("Training Loss vs. Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Average Loss")
    plt.grid(True)
    plt.show()

    # This will now print cleanly on a new line
    print(f"Epoch {epoch + 1} completed. Average Loss: {avg_loss:.5f}")

Starting training...


Epoch 1/5:   0%|          | 2/12608 [00:02<4:17:00,  1.22s/it, Loss=0.99887]


OutOfMemoryError: CUDA out of memory. Tried to allocate 500.00 MiB. GPU 0 has a total capacity of 15.45 GiB of which 309.75 MiB is free. Process 2824371 has 783.70 MiB memory in use. Process 2848167 has 8.47 GiB memory in use. Including non-PyTorch memory, this process has 5.65 GiB memory in use. Of the allocated memory 3.75 GiB is allocated by PyTorch, and 1.57 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# 6. Save Checkpoint
# We save the model's state_dict, which now securely contains the data_mean
# and data_std buffers required for inference.
save_path = "kolmogorov_diffusion_model.pt"
torch.save(
    {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": avg_loss,
    },
    save_path,
)
print(f"Checkpoint saved to {save_path}")
print("-" * 50)